In [1]:
import os
import xarray as xr
from datetime import datetime
import numpy as np
from netCDF4 import Dataset
import pandas as pd

# Python libraries for visualization
import matplotlib.pyplot as plt
import matplotlib.colors
import matplotlib.cm as cm
from matplotlib.axes import Axes
import cartopy
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.geoaxes import GeoAxes
from cartopy.io.shapereader import Reader
import cartopy.feature as cfeat
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import scipy.interpolate.ndgriddata as ndgriddata
GeoAxes._pcolormesh_patched = Axes.pcolormesh
from skimage import exposure
import cartopy.feature as cfeature
import rasterio
import pyproj

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action = "ignore", category = RuntimeWarning)
import scipy.io
import mat73


C:\Users\varun.katoch\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [2]:
##https://gitlab.eumetsat.int/eumetlab/atmosphere/atmosphere/-/blob/master/functions.ipynb#generate_xr_from_1D_vec
%run D:/NH3_IASI_Data/functions.ipynb

In [3]:
mat=mat73.loadmat(r"D:\NH3_IASI_Data\12km.mat")
lat_x=mat['lat']
lon_x=mat['lon']

In [5]:
from datetime import date, timedelta
from netCDF4 import Dataset, date2num
import numpy as np
import datetime as dt
import xarray as xr
import pyproj

# Function to generate date range
def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days)):
        yield start_date + timedelta(n)

# Open a new NetCDF file for writing
ncfile = Dataset('D:/NH3_IASI_Data/data/IASI_METOPC_4R_2019_2023.nc', mode='w', format='NETCDF4')

# Create dimensions
lat_dim = ncfile.createDimension('lat', 148)     # latitude axis
lon_dim = ncfile.createDimension('lon', 175)     # longitude axis
time_dim = ncfile.createDimension('time', None)  # unlimited axis for time

# Create global attributes
ncfile.title = 'My model data'
ncfile.subtitle = 'IASI METOP A V4r'

# Create variables
lat = ncfile.createVariable('lat', np.float32, ('lat',))
lat.units = 'degrees_north'
lat.long_name = 'latitude'

lon = ncfile.createVariable('lon', np.float32, ('lon',))
lon.units = 'degrees_east'
lon.long_name = 'longitude'

time = ncfile.createVariable('time', np.float64, ('time',))
time.units = 'hours since 1800-01-01'
time.long_name = 'time'

nh3 = ncfile.createVariable('NH3', np.float64, ('time', 'lat', 'lon'))
nh3.units = 'molecules/cm² x E16'  # or whatever the correct units are
nh3.standard_name = 'ammonia'  # CF standard name

# Define the projection using pyproj
crs = pyproj.CRS('EPSG:4326')
ncfile.crs = crs.to_wkt()  # Optional: write CRS to the file

# Set latitude and longitude values
lat[:] = np.unique(lat_x)  # Flatten if necessary, here we assume unique lat values
lon[:] = np.unique(lon_x)  # Flatten if necessary, here we assume unique lon values

# # Loop through years and dates
# Set start date for testing
# start_date = date(2008,6,1)  # Adjust this to your preferred start date
# end_date = start_date + timedelta(days=30)  # 10 days duration
for y in range(2019,2024):
    start_date = date(int(f'{y}'),1, 1)
    end_date = date(int(f'{y+1}'),1, 1)
    for single_date in daterange(start_date, end_date):
        try:
            # Open the dataset for the specific date
            data = xr.open_dataset(f'D:/NH3_IASI_Data/data/METOP-C/{single_date.strftime("%Y")}/IASI_METOPC_L2_NH3_{single_date.strftime("%Y")}{single_date.strftime("%m")}{single_date.strftime("%d")}_ULB-LATMOS_V4.0.0R.nc')

            nh3_data = data['nh3_total_column']
            iasi_co_da = generate_xr_from_1D_vec(file=data,
                                                   lat_path='latitude',
                                                   lon_path='longitude',
                                                   variable=nh3_data.data,
                                                   parameter_name=nh3_data.name,
                                                   longname=nh3_data.long_name,
                                                   no_of_dims=1,
                                                   unit=nh3_data.units)

            qf = data['postfilter']
            iasi_co_qf_da = generate_xr_from_1D_vec(file=data,
                                                      lat_path='latitude',
                                                      lon_path='longitude',
                                                      variable=qf.data,
                                                      parameter_name=qf.name,
                                                      longname=qf.long_name,
                                                      no_of_dims=1,
                                                      unit='-')

            iasi_co_masked = generate_masked_array(xarray=iasi_co_da,
                                                   mask=iasi_co_qf_da,
                                                   threshold=1,
                                                   operator='=',
                                                   drop=False)

            PM = data['AMPM']
            iasi_day = generate_xr_from_1D_vec(file=data,
                                                 lat_path='latitude',
                                                 lon_path='longitude',
                                                 variable=PM.data,
                                                 parameter_name=PM.name,
                                                 longname=PM.long_name,
                                                 no_of_dims=1,
                                                 unit='-')

            iasi_nh3_masked_day = generate_masked_array(xarray=iasi_co_masked,
                                                         mask=iasi_day,
                                                         threshold=0,
                                                         operator='=',
                                                         drop=False)

            iasi_nh3_converted = iasi_nh3_masked_day * nh3_data.multiplication_factor_to_convert_to_molecules_per_cm2

            cloudCov = data['cloud_coverage']
            nh3_mask_2a = generate_xr_from_1D_vec(file=data,
                                                    lat_path='latitude',
                                                    lon_path='longitude',
                                                    variable=cloudCov,
                                                    parameter_name=cloudCov.name,
                                                    longname=cloudCov.long_name,
                                                    no_of_dims=1,
                                                    unit=cloudCov.units)

            nh3_2a_masked = generate_masked_array(xarray=iasi_nh3_converted,
                                                   mask=nh3_mask_2a,
                                                   threshold=0.25,
                                                   operator='<',
                                                   drop=False)

           
            df = nh3_2a_masked.data * 1e-16
            longitude = nh3_2a_masked.longitude.data
            latitude = nh3_2a_masked.latitude.data
            test = ndgriddata.griddata((longitude, latitude), df, (lon_x, lat_x), method='nearest')

            # Get the time index for the current date
            time_index = ncfile.dimensions['time'].size  # Current size of the time dimension

            # Convert single_date to a datetime object
            single_datetime = dt.datetime(single_date.year, single_date.month, single_date.day)

            # Convert to time value
            time_value = date2num(single_datetime, time.units)

            new_data_rot = np.rot90(test, k=1)
            new_data_flip = np.flipud(new_data_rot)

            # Assign data to the NetCDF variable
            nh3[time_index, :, :] = new_data_flip  # Populate NH3 data

            # Update the time variable
            time[time_index] = time_value
            print(single_date)
            del data
            del new_data_rot
            del new_data_flip
            del test

        except Exception as e:
            print(f'This Date of Data is not Present {single_date}: {e}')
            pass

# Close the NetCDF file
ncfile.close()
print('Dataset is closed!')


This Date of Data is not Present 2019-01-01: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190101_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-01-02: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190102_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-01-03: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190103_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-01-04: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190104_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-01-05: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190105_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-01-06: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_ME

This Date of Data is not Present 2019-03-26: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190326_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-03-27: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190327_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-03-28: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190328_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-03-29: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190329_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-03-30: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190330_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-03-31: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_ME

This Date of Data is not Present 2019-06-10: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190610_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-06-11: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190611_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-06-12: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190612_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-06-13: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190613_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-06-14: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190614_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-06-15: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_ME

This Date of Data is not Present 2019-08-19: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190819_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-08-20: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190820_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-08-21: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190821_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-08-22: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190822_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-08-23: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_METOPC_L2_NH3_20190823_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 2019-08-24: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2019\\IASI_ME

2020-03-03
2020-03-04
2020-03-05
2020-03-06
2020-03-07
2020-03-08
2020-03-09
2020-03-10
2020-03-11
2020-03-12
2020-03-13
2020-03-14
2020-03-15
2020-03-16
2020-03-17
2020-03-18
2020-03-19
2020-03-20
2020-03-21
2020-03-22
2020-03-23
2020-03-24
2020-03-25
2020-03-26
2020-03-27
2020-03-28
2020-03-29
2020-03-30
2020-03-31
2020-04-01
2020-04-02
2020-04-03
2020-04-04
2020-04-05
2020-04-06
2020-04-07
2020-04-08
2020-04-09
2020-04-10
2020-04-11
2020-04-12
2020-04-13
2020-04-14
2020-04-15
2020-04-16
2020-04-17
2020-04-18
2020-04-19
2020-04-20
2020-04-21
2020-04-22
2020-04-23
2020-04-24
2020-04-25
2020-04-26
2020-04-27
2020-04-28
2020-04-29
2020-04-30
2020-05-01
2020-05-02
2020-05-03
2020-05-04
2020-05-05
2020-05-06
2020-05-07
2020-05-08
2020-05-09
2020-05-10
2020-05-11
2020-05-12
2020-05-13
2020-05-14
2020-05-15
2020-05-16
2020-05-17
2020-05-18
2020-05-19
2020-05-20
2020-05-21
2020-05-22
2020-05-23
2020-05-24
2020-05-25
2020-05-26
2020-05-27
2020-05-28
2020-05-29
2020-05-30
2020-05-31
2020-06-01

2022-01-09
2022-01-10
2022-01-11
2022-01-12
2022-01-13
2022-01-14
2022-01-15
2022-01-16
2022-01-17
2022-01-18
2022-01-19
2022-01-20
2022-01-21
2022-01-22
2022-01-23
2022-01-24
2022-01-25
2022-01-26
2022-01-27
2022-01-28
2022-01-29
2022-01-30
2022-01-31
2022-02-01
2022-02-02
2022-02-03
2022-02-04
2022-02-05
2022-02-06
2022-02-07
2022-02-08
2022-02-09
2022-02-10
2022-02-11
2022-02-12
2022-02-13
2022-02-14
2022-02-15
2022-02-16
2022-02-17
2022-02-18
2022-02-19
2022-02-20
2022-02-21
2022-02-22
2022-02-23
2022-02-24
2022-02-25
2022-02-26
2022-02-27
2022-02-28
2022-03-01
2022-03-02
2022-03-03
2022-03-04
2022-03-05
2022-03-06
2022-03-07
2022-03-08
2022-03-09
2022-03-10
2022-03-11
2022-03-12
2022-03-13
2022-03-14
2022-03-15
2022-03-16
2022-03-17
2022-03-18
2022-03-19
2022-03-20
2022-03-21
2022-03-22
2022-03-23
2022-03-24
2022-03-25
2022-03-26
2022-03-27
2022-03-28
2022-03-29
2022-03-30
2022-03-31
2022-04-01
2022-04-02
2022-04-03
2022-04-04
2022-04-05
2022-04-06
2022-04-07
2022-04-08
2022-04-09

2023-10-15
2023-10-16
2023-10-17
2023-10-18
2023-10-19
2023-10-20
2023-10-21
2023-10-22
2023-10-23
2023-10-24
2023-10-25
2023-10-26
2023-10-27
2023-10-28
2023-10-29
2023-10-30
2023-10-31
2023-11-01
2023-11-02
2023-11-03
2023-11-04
2023-11-05
2023-11-06
2023-11-07
2023-11-08
2023-11-09
2023-11-10
2023-11-11
2023-11-12
2023-11-13
2023-11-14
2023-11-15
2023-11-16
2023-11-17
2023-11-18
2023-11-19
2023-11-20
2023-11-21
2023-11-22
2023-11-23
2023-11-24
2023-11-25
2023-11-26
2023-11-27
2023-11-28
2023-11-29
This Date of Data is not Present 2023-11-30: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2023\\IASI_METOPC_L2_NH3_20231130_ULB-LATMOS_V4.0.0R.nc'
2023-12-01
2023-12-02
2023-12-03
2023-12-04
2023-12-05
2023-12-06
2023-12-07
2023-12-08
2023-12-09
2023-12-10
This Date of Data is not Present 2023-12-11: [Errno 2] No such file or directory: 'D:\\NH3_IASI_Data\\data\\METOP-C\\2023\\IASI_METOPC_L2_NH3_20231211_ULB-LATMOS_V4.0.0R.nc'
This Date of Data is not Present 202